In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.dates as mdates

In [67]:
from scipy.signal import find_peaks

def make_zoom_plot(xmin, xmax, ymin=0, ymax=150, filename="zoom.png", title="Zoomed CH$_4$ Region", line_color="tab:blue"):
    fig, ax = plt.subplots(figsize=(8, 4.8))

    # Plot full data, then zoom with xlim
    ax.plot(df["Elapsed_sec"], df["CH4_ppm"], linewidth=1.8, label="CH$_4$", color=line_color)
    ax.scatter(df["Elapsed_sec"], df["CH4_ppm"], s=14, alpha=0.7, color=line_color)

    # Threshold line
    ax.axhline(
        y=100,
        linestyle="--",
        linewidth=1.5,
        color="red",
        label="100 ppm threshold"
    )

    # Threshold text
    ax.text(
        xmin + 0.72 * (xmax - xmin),
        102,
        "100 ppm threshold",
        color="red",
        ha="left",
        va="bottom",
        fontsize=10
    )

    # Zoom window
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Select points in this zoom region
    zoom_df = df[(df["Elapsed_sec"] >= xmin) & (df["Elapsed_sec"] <= xmax)].copy()

    # Find actual peaks above 2.1 ppm
    peak_indices, properties = find_peaks(
    zoom_df["CH4_ppm"].values,
    height=2.150,
    prominence=0.05,
    distance=1
    )

    label_points = zoom_df.iloc[peak_indices].copy()
    label_points = label_points.sort_values("Elapsed_sec")

    labels_text = []

    for i, (_, row) in enumerate(label_points.iterrows(), start=1):
        key = f"P{i}"

        ax.axvline(
            row["Elapsed_sec"],
            linestyle=":",
            linewidth=1.2,
            color="black",
            alpha=0.8
        )

        y_text = ymax - 8 - ((i - 1) % 2) * 6

        ax.text(
            row["Elapsed_sec"],
            y_text,
            key,
            ha="center",
            va="bottom",
            fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=0.2)
        )

        labels_text.append(
            f"{key} = ({row['CH4_ppm']:.2f} ppm, {row['Datetime'].strftime('%H:%M:%S')})"
        )

    if labels_text:
        ax.text(
            1.02, 0.98,
            "\n".join(labels_text),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
        )

    ax.set_xlabel("Elapsed Time (sec)")
    ax.set_ylabel("CH$_4$ (ppm)")
    ax.set_title(title)

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.spines["top"].set_linewidth(1.0)
    ax.spines["right"].set_linewidth(1.0)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)

    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(frameon=False)

    plt.subplots_adjust(right=0.78)
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()

In [68]:
csv_file = '/Users/erumhassan/Library/CloudStorage/OneDrive-UniversityofOklahoma/Caddo Nation Field Survey - Documents/Methane_Data/Licor_7810/04_09_2026/Well16.csv'          # change to your CSV filename
output_table_file = "processed_ch4_data.csv"
output_figure_file = "seconds_vs_ch4.png"

In [69]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv(csv_file, encoding="cp1252")

# Clean column names
df.columns = df.columns.str.strip()

# Print columns so you can verify names
print("Columns found:", df.columns.tolist())

# Make sure required columns exist
required_cols = ["SECONDS", "DATE", "TIME", "CH4"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# =========================
# CLEAN / CONVERT COLUMNS
# =========================
df["SECONDS"] = pd.to_numeric(df["SECONDS"], errors="coerce")
df["CH4"] = pd.to_numeric(df["CH4"], errors="coerce")

# Combine DATE + TIME into one datetime column
df["Datetime"] = pd.to_datetime(
    df["DATE"].astype(str).str.strip() + " " + df["TIME"].astype(str).str.strip(),
    errors="coerce"
)

# Convert CH4 from ppb to ppm
df["CH4_ppm"] = df["CH4"] / 1000.0

# Keep only valid rows
df = df.dropna(subset=["SECONDS", "Datetime", "CH4_ppm"]).copy()

# Sort by seconds
df = df.sort_values("SECONDS")

# Make elapsed time start at zero
df["Elapsed_sec"] = df["SECONDS"] - df["SECONDS"].iloc[0]

# =========================
# BASELINE / AVERAGE
# =========================
baseline_ch4 = df.loc[(df["Elapsed_sec"] >= 0) & (df["Elapsed_sec"] <= 180), "CH4_ppm"].mean()
print(f"Average CH4 from 0–60 s: {baseline_ch4:.3f} ppm")

Columns found: ['DATAH', 'SECONDS', 'NANOSECONDS', 'NDX', 'DIAG', 'REMARK', 'DATE', 'TIME', 'H2O', 'CO2', 'CH4', 'CAVITY_P', 'CAVITY_T', 'LASER_PHASE_P', 'LASER_T', 'RESIDUAL', 'RING_DOWN_TIME', 'THERMAL_ENCLOSURE_T', 'PHASE_ERROR', 'LASER_T_SHIFT', 'INPUT_VOLTAGE', 'CHK']
Average CH4 from 0–60 s: 2.150 ppm


/var/folders/dh/kdngnmhn0f736rc0pkkrdt8h0000gn/T/ipykernel_13074/11540182.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Datetime"] = pd.to_datetime(


In [78]:
# PLOT
plt.rcParams.update({
    "font.size": 11,
    "axes.linewidth": 1.0
})

fig, ax = plt.subplots(figsize=(10, 5.5))

ax.plot(df["Elapsed_sec"], df["CH4_ppm"], linewidth=1.8, label="CH$_4$", color="#4C9F9B")
ax.scatter(df["Elapsed_sec"], df["CH4_ppm"], s=14, alpha=0.7, color="#4C9F9B")

ax.set_xlabel("Elapsed Time (sec)")
ax.set_ylabel("CH$_4$ (ppm)")
ax.set_ylim(0, 150)
ax.set_title("CH$_4$ Concentration vs Time")


ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(frameon=False)

#setting the device limit 
ax.axhline(
    y=100,
    linestyle="--",
    linewidth=1.5,
    color="red",
    label="100 ppm threshold"
)
ax.text(
    df["Elapsed_sec"].max() * 0.98,
    102,
    "100 ppm threshold",
    color="red",
    ha="right",
    va="bottom",
    fontsize=10
)

make_zoom_plot(220, 420, filename="zoom_region_1.png", title="Zoomed Region: 220–420 sec", line_color="#4C9F9B")
make_zoom_plot(420, 980, filename="zoom_region_2.png", title="Zoomed Region: 420–980 sec", line_color="#C98C2B")
make_zoom_plot(980, 1040, filename="zoom_region_3.png", title="Zoomed Region: 980–1040 sec", line_color="#B94E48")
make_zoom_plot(1210, 1245, filename="zoom_region_4.png", title="Zoomed Region: 1210–1245 sec", line_color="#8E5C99")w

plt.tight_layout()
#plt.savefig(output_figure_file, dpi=300, bbox_inches="tight")
plt.show()

#print(f"Figure saved as: {output_figure_file}")

SyntaxError: invalid syntax (2297016381.py, line 42)